# <font color="blue"> Cuaderno 14: Autoencoders en Redes Neuronales </font>

# Introducción a los Autoencoders

Este cuaderno tiene como objetivo introducir el concepto de **autoencoders**, explicar cómo se construyen utilizando redes neuronales y documentar tres ejemplos prácticos:

- Autoencoder básico  
- Eliminación de ruido en imágenes  
- Detección de anomalías  



## ¿Qué son los Autoencoders?

Los **autoencoders** son un tipo de red neuronal que pertenece al campo del **aprendizaje no supervisado**. Su principal objetivo es aprender una representación eficiente y comprimida de los datos de entrada mediante un proceso de **codificación y decodificación**.

### Componentes principales:

- **Codificador (Encoder):** comprime la entrada en una representación de menor dimensión conocida como *espacio latente*.
- **Decodificador (Decoder):** reconstruye la entrada original a partir de esa representación comprimida.

-

## ¿Para qué se utilizan?

Los autoencoders se aplican en tareas como:

- Reducción de dimensionalidad  
- Compresión de datos  
- Eliminación de ruido  
- Detección de anomalías  
- Generación de datos sintéticos (en variantes como los *Variational Autoencoders*)



## ¿Cómo se implementan con redes neuronales?

Un autoencoder está compuesto por dos bloques principales:

### 🔹 Codificador (Encoder)

Una serie de capas (densas, convolucionales, etc.) que transforman los datos de entrada en una representación más compacta. Esta salida se conoce como **espacio latente (latent space)**, y es clave porque captura las características más relevantes del dato original.

### 🔹 Decodificador (Decoder)

Toma el vector del espacio latente y trata de **reconstruir la entrada original**. El entrenamiento se basa en minimizar la diferencia entre la entrada y la salida reconstruida, generalmente usando **Mean Squared Error (MSE)** como función de pérdida.


## Arquitectura básica

Un autoencoder básico puede construirse usando capas densas (*fully connected*), pero también pueden usarse **redes convolucionales (CNNs)** para datos como imágenes, donde es importante conservar la estructura espacial.

---

## Espacio Latente (Latent Space)

El **espacio latente** es la representación comprimida que produce el codificador. Esta representación:

- Reduce la dimensionalidad de los datos  
- Conserva la información más importante  
- Permite detectar patrones y anomalías  
- Puede visualizarse en 2D/3D usando técnicas como **t-SNE** o **PCA**

Además, puede utilizarse para tareas avanzadas como interpolación entre ejemplos o generación de nuevas muestras similares.

---


Un autoencoder básico puede consistir en capas densas (fully connected layers), pero también se pueden usar otras estructuras como redes convolucionales para tareas más complejas.

![](https://contenthub-static.grammarly.com/blog/wp-content/uploads/2024/10/6303_blog-visuals-auto-encoders_1500X800.png)
---

## Usos y Ventajas de los Autoencoders

1. **Reducción de Dimensionalidad**: Similar al Análisis de Componentes Principales (PCA), los autoencoders pueden reducir las dimensiones de los datos mientras retienen la mayor parte de la información.
2. **Compresión de Datos**: Pueden comprimir datos y almacenar solo la representación compacta.
3. **Eliminación de Ruido**: Se pueden entrenar para eliminar ruido de las imágenes, como veremos más adelante.
4. **Detección de Anomalías**: Los autoencoders pueden detectar patrones inusuales o outliers al comparar la entrada con la salida reconstruida.
5. **Generación de Datos**: Autoencoders como los Variational Autoencoders (VAEs) pueden generar nuevos ejemplos de datos.



Un ejemplo de diseño de la red neruonal Autoencoder es la siguiente:

```python
from keras.models import Model
from keras.layers import Input, Dense, BatchNormalization, Dropout
from keras.optimizers import Adam
from keras import regularizers

# Parámetros
input_dim = 784
encoding_dim = 32
reg = regularizers.l2(1e-5)

# Codificador
input_img = Input(shape=(input_dim,))
x = Dense(256, activation='relu', kernel_regularizer=reg)(input_img)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
x = Dense(128, activation='relu', kernel_regularizer=reg)(x)
x = Dense(64, activation='relu', kernel_regularizer=reg)(x)
encoded = Dense(encoding_dim, activation='relu', name='latent_space')(x)

# Decodificador
x = Dense(64, activation='relu', kernel_regularizer=reg)(encoded)
x = Dense(128, activation='relu', kernel_regularizer=reg)(x)
x = BatchNormalization()(x)
decoded = Dense(input_dim, activation='sigmoid')(x)

# Autoencoder completo
autoencoder = Model(inputs=input_img, outputs=decoded, name='autoencoder')
autoencoder.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy')

# También podemos definir el encoder y decoder por separado
encoder = Model(inputs=input_img, outputs=encoded, name='encoder')

encoded_input = Input(shape=(encoding_dim,))
decoder_layer = autoencoder.layers[-3](encoded_input)
for layer in autoencoder.layers[-2:]:
    decoder_layer = layer(decoder_layer)
decoder = Model(encoded_input, decoder_layer, name='decoder')

autoencoder.summary()
```


## <font color="blue"> Ejemplo 1: Autoencoder Básico conjunto de datos Fashion MNIST</font>

A continuación, implementaremos un autoencoder básico utilizando el conjunto de datos **Fashion MNIST**. Este conjunto contiene imágenes de 28x28 píxeles de diferentes tipos de ropa.

1. Primero cargamos los datos y normalizamos las imágenes.
2. Creamos el modelo con un codificador y decodificador simples.
3. Entrenamos el modelo y mostramos cómo el autoencoder reconstruye las imágenes.


## Importar TensorFlow y otras bibliotecas

In [ ]:
# =====================================================
# 1️ IMPORTACIÓN DE LIBRERÍAS
# =====================================================
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import losses
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, UpSampling2D,
    Dense, Flatten, Reshape, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import fashion_mnist


## Cargue el conjunto de datos
Para comenzar, entrenará el codificador automático básico utilizando el conjunto de datos Fashion MNIST. Cada imagen en este conjunto de datos tiene 28x28 píxeles.

. Se Cargaron en el dataset de fashion_mnist, con  60,000 imágenes de entrenamiento y 10,000 de prueba.

* Se ignoran las etiquetas (_) porque el autoencoder no necesita saber qué clase es cada imagen, solo quiere reconstruirlas.

* **Normaliza los datos:**

Convierte de enteros uint8 (0-255) a float32 y los escala a valores entre 0.0 y 1.0. Esto es esencial para entrenar redes neuronales de forma eficiente.

* **Imprime las formas (shapes) de los datos:**

x_train.shape = (60000, 28, 28)
x_test.shape = (10000, 28, 28)

Esto confirma que tienes 60k imágenes de entrenamiento y 10k de prueba, cada una de 28×28 píxeles.



In [ ]:
# =====================================================
# 2️ CARGA Y PREPROCESAMIENTO DE DATOS
# (solo una vez en todo el flujo)
# =====================================================
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalizamos y redimensionamos solo aquí
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

# Añadimos dimensión de canal (de 28x28 → 28x28x1)
x_train = np.expand_dims(x_train, axis=-1)
x_test  = np.expand_dims(x_test, axis=-1)

print("Tamaño de entrenamiento:", x_train.shape)
print("Tamaño de prueba:", x_test.shape)


In [ ]:
# Mostrar las primeras 5 etiquetas con sus nombres


class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

for i in range(5):
    print(f"Etiqueta {y_train[i]} → {class_names[y_train[i]]}")


In [ ]:
import matplotlib.pyplot as plt

# Mostrar las primeras 5 imágenes con sus nombres
plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(class_names[y_train[i]])
    plt.axis('off')
    plt.tight_layout()
    
plt.show()


#Autoencoder híbrido (Conv2D + Dense en el centro)

Se usa Conv2D, MaxPooling2D, UpSampling2D si se está trabajando directamente con imágenes.

Solo se usa Dense si necesitas compactar a un vector o hacer algo especial en el latent space.

Lo ideal muchas veces es un híbrido: convolucional al inicio/final, y denso en el centro (latente).

Este es el ejemplo que vamos hacer:
![](https://github.com/adiacla/bigdata/blob/master/autoencoderImg.png?raw=true)


## Explicación del Autoencoder Híbrido (Conv2D + Dense)

Este modelo es un autoencoder diseñado para imágenes de tamaño **28x28x1** (por ejemplo, del dataset Fashion MNIST). Se compone de tres partes principales: **Encoder, Vector Latente y Decoder**. Usa tanto capas convolucionales como densas para aprender representaciones compactas.



### **Encoder**
El encoder convierte la imagen en una representación comprimida:

- `Conv2D(32)` → extrae 32 mapas de características con filtro 3x3.
- `MaxPooling2D` → reduce resolución a la mitad (28x28 → 14x14).
- `Conv2D(64)` → extrae características más profundas.
- `MaxPooling2D` → reduce resolución a 7x7, manteniendo 64 canales.



### **Latent Space (Vector latente)**
- `Flatten` → convierte el tensor 7x7x64 en un vector de 3136 elementos.
- `Dense(64)` → capa densa que reduce la información a un vector de **64 dimensiones**, que representa la **codificación comprimida** de la imagen.



### **Decoder**
Reconstruye la imagen original desde el vector latente:

- `Dense(3136)` → expande el vector a su forma original.
- `Reshape(7x7x64)` → lo vuelve a transformar a una estructura espacial.
- `UpSampling2D` → aumenta la resolución de nuevo a 14x14, luego a 28x28.
- `Conv2D(64)` y `Conv2D(32)` → refinan la reconstrucción.
- `Conv2D(1, activation='sigmoid')` → genera la imagen final reconstruida (28x28x1), con valores entre 0 y 1.



### **Compilación**
El modelo se compila con:
- `optimizer='adam'`: optimizador eficiente y adaptativo.
- `loss='binary_crossentropy'`: apropiado para reconstrucción de imágenes binarizadas o normalizadas entre 0 y 1.



Este diseño **híbrido** permite una representación eficiente y flexible, combinando lo mejor de las capas convolucionales (estructura espacial) y densas (compresión global).


In [ ]:
# Entrada de imagen (28x28x1)
entrada = Input(shape=(28, 28, 1), name="entrada")

# --- ENCODER ---
x = Conv2D(32, (3, 3), activation='relu', padding='same')(entrada)     # 28x28x32
x = MaxPooling2D((2, 2), padding='same')(x)                            # 14x14x32
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)          # 14x14x64
x = MaxPooling2D((2, 2), padding='same')(x)                            # 7x7x64

# A vector (Flatten) + Dense (cuello de botella)
x = Flatten()(x)                                                      # 7*7*64 = 3136
encoded = Dense(64, activation='relu', name='latent_vector')(x)       # Vector latente

# --- DECODER ---
x = Dense(7*7*64, activation='relu')(encoded)                         # Expansión
x = Reshape((7, 7, 64))(x)                                            # De vuelta a 3D
x = UpSampling2D((2, 2))(x)                                           # 14x14x64
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)                                           # 28x28x64
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)

# Salida final (una imagen 28x28x1)
salida = Conv2D(1, (3, 3), activation='sigmoid', padding='same', name='salida')(x)

# Definir modelo
autoencoder = Model(inputs=entrada, outputs=salida)

autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

In [ ]:
# Autoencoder
autoencoder.summary()

Entrene el modelo usando x_train como entrada y como destino. El encoder aprenderá a comprimir el conjunto de datos de 784 dimensiones al espacio latente, y el decoder aprenderá a reconstruir las imágenes originales.
.

In [ ]:
# =====================================================
# 4️ ENTRENAMIENTO (usa los datos ya normalizados)
# =====================================================
history = autoencoder.fit(
    x_train, x_train,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test)
)


In [ ]:
#Guardar el model de keras
autoencoder.save('autoencoder.keras')

### El siguiente código muestra las imagenes que el puede reconstruir con las entradas de los datos de prueba

In [ ]:
# =====================================================
# 5️ RECONSTRUCCIÓN Y VISUALIZACIÓN
# =====================================================
decoded_imgs = autoencoder.predict(x_test)

n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
    # Original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    plt.title("Original")
    plt.axis("off")

    # Reconstruida
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap='gray')
    plt.title("Reconstruida")
    plt.axis("off")
plt.show()

Ahora que el modelo está entrenado, probémoslo codificando y decodificando imágenes del conjunto de prueba.


* Codificar las imágenes de prueba (esto extrae las características de las imágenes en el espacio latente).

* Decodificar las imágenes codificadas (esto reconstruye las imágenes a partir de su representación comprimida).

* Mostrar las imágenes originales y las reconstruidas para compararlas.

## <font color="blue"> Ejemplo 2: Eliminación de Ruido en Imágenes </font>

![](https://github.com/adiacla/bigdata/blob/master/autoencoderImgruido.png?raw=true)

En este taller, crearemos un autoencoder que eliminará el ruido de las imágenes de Fashion MNIST.

    Agregamos ruido a las imágenes.
    Creamos un autoencoder convolucional para reconstruir las imágenes originales.
    Comprobamos los resultados.

![Image denoising results](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/generative/images/image_denoise_fmnist_results.png?raw=1)

Vamos a entrenar un codificador automático para eliminar el ruido de las imágenes. En la siguiente sección, creará una versión ruidosa del conjunto de datos Fashion MNIST aplicando ruido aleatorio a cada imagen. Luego entrenará un codificador automático utilizando la imagen ruidosa como entrada y la imagen original como objetivo.

Volvamos a importar el conjunto de datos para omitir las modificaciones realizadas anteriormente.

Agregar ruido aleatorio a las imágenes

Este código simula imágenes con ruido, añadiendo ruido gaussiano controlado a las imágenes originales. Es útil para entrenar autoencoders en tareas de eliminación de ruido (denoising autoencoders), donde el modelo aprende a restaurar imágenes limpias a partir de versiones ruidosas.


Este proceso de añadir ruido es una técnica común en el aprendizaje no supervisado, ya que mejora la capacidad del modelo para generalizar y hacer predicciones más robustas en datos ruidosos.


La operación tf.clip_by_value() asegura que los valores de los píxeles ruidosos estén dentro del rango válido para imágenes normalizadas (0 a 1).

Si algún píxel tiene un valor fuera de este rango debido al ruido, se recorta para que quede dentro de los límites establecidos (clip_value_min=0. y clip_value_max=1.).

In [ ]:
noise_factor = 0.3
x_train_noisy = x_train + noise_factor * tf.random.normal(shape=x_train.shape)
x_test_noisy = x_test + noise_factor * tf.random.normal(shape=x_test.shape)

x_train_noisy = tf.clip_by_value(x_train_noisy, clip_value_min=0., clip_value_max=1.)
x_test_noisy = tf.clip_by_value(x_test_noisy, clip_value_min=0., clip_value_max=1.)

Veamos las imágenes ruidosas.

In [ ]:
n = 10
plt.figure(figsize=(20, 2))
for i in range(n):
    ax = plt.subplot(1, n, i + 1)
    plt.imshow(tf.squeeze(x_test_noisy[i]), cmap='gray')  # mejor forma de asegurar escala
    plt.title("Ruido")
    plt.axis("off")
plt.suptitle("Ejemplos de imágenes con ruido", fontsize=14)
plt.show()


In [ ]:
n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
    # Original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(tf.squeeze(x_test[i]), cmap='gray')
    plt.title("Original")
    plt.axis("off")

    # Con ruido
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(tf.squeeze(x_test_noisy[i]), cmap='gray')
    plt.title("Con ruido")
    plt.axis("off")

plt.suptitle("Comparación: Imágenes originales vs con ruido", fontsize=14)
plt.show()


### Definir un codificador automático convolucional

Definir un codificador automático convolucional
En este ejemplo, entrenará un codificador automático convolucional utilizando capas Conv2D en el encoder y capas Conv2DTranspose en el decoder .

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Dense, Flatten, Reshape
from tensorflow.keras.models import Model

# Entrada de imagen (28x28x1)
entrada = Input(shape=(28, 28, 1), name="entrada")

# --- ENCODER ---
x = Conv2D(32, (3, 3), activation='relu', padding='same')(entrada)     # 28x28x32
x = MaxPooling2D((2, 2), padding='same')(x)                            # 14x14x32
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)          # 14x14x64
x = MaxPooling2D((2, 2), padding='same')(x)                            # 7x7x64

# A vector (Flatten) + Dense (cuello de botella)
x = Flatten()(x)                                                      # 7*7*64 = 3136
encoded = Dense(64, activation='relu', name='latent_vector')(x)       # Vector latente

# --- DECODER ---
x = Dense(7*7*64, activation='relu')(encoded)                         # Expansión
3
x = Reshape((7, 7, 64))(x)                                            # De vuelta a 3D
x = UpSampling2D((2, 2))(x)                                           # 14x14x64
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = UpSampling2D((2, 2))(x)                                           # 28x28x64
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)

# Salida final (una imagen 28x28x1)
salida = Conv2D(1, (3, 3), activation='sigmoid', padding='same', name='salida')(x)

# Definir modelo
autoencoderruido = Model(inputs=entrada, outputs=salida)

autoencoderruido.compile(optimizer='adam', loss='binary_crossentropy')

In [ ]:
autoencoderruido.summary()


In [ ]:
history=autoencoderruido.fit(
    x=x_train_noisy,              # Imágenes con ruido como entrada
    y=x_train,                    # Imágenes originales como objetivo
    epochs=20,                    # Número de épocas
    batch_size=128,               # Tamaño del lote
    shuffle=True,                 # Mezcla aleatoria en cada época
    validation_data=(x_test_noisy, x_test),  # Validación: ruido como entrada, imagen limpia como salida
    verbose=1                     # Muestra progreso del entrenamiento
)


In [ ]:
decoded_imgs_ruido = autoencoderruido.predict(x_test_noisy)

n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    # Con ruido
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(tf.squeeze(x_test_noisy[i]), cmap='gray')
    plt.title("Con ruido")
    plt.axis('off')

    # Reconstruida
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(decoded_imgs_ruido[i].reshape(28, 28), cmap='gray')
    plt.title("Reconstruida")
    plt.axis('off')

    # Original
    ax = plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    plt.title("Original")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Evolución del Autoencoder Denoising')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.show()